# Titanic — v6: Hyperparameter Search Loop

**Version 6.** Keeps what worked (group survival rate, title-median age imputation, regularization), removes what didn't (FarePP, title×class imputation), and adds a **random-search loop** over gradient-boosting hyperparameters that does not stop on streaks without improvement — the best config appeared at iteration 19, after 11 consecutive misses.

Search space: learning_rate {0.01–0.08}, n_estimators {200–800}, max_depth {2–4}, subsample {0.6–1.0}, min_samples_leaf {1–20}. Winner: **lr 0.01, 500 trees, depth 3, subsample 0.6, min_samples_leaf 3** → CV **0.8530** (v5: 0.8496).


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

def add_features(df):
    df = df.copy()
    df["Title"] = df.Name.str.extract(r",\s*([^.]+)\.")[0].str.strip().replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["Title"] = df.Title.where(df.Title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    df["FamilySize"] = df.SibSp + df.Parch + 1
    df["IsAlone"] = (df.FamilySize == 1).astype(int)
    df["HasCabin"] = df.Cabin.notna().astype(int)
    df["Surname"] = df.Name.str.split(",").str[0]
    df["GroupID"] = df.Surname + "_" + df.Ticket.str.replace(r"\D", "", regex=True).str[:-1].fillna("")
    return df

train_fe = add_features(train)
test_fe = add_features(test)

age_by_title = train_fe.groupby("Title").Age.median()
fare_by_class = train_fe.groupby("Pclass").Fare.median()
for df in (train_fe, test_fe):
    df["Age"] = df.Age.fillna(df.Title.map(age_by_title))
    df["Fare"] = df.Fare.fillna(df.Pclass.map(fare_by_class))
    df["Embarked"] = df.Embarked.fillna("S")

grp = train_fe.groupby("GroupID").Survived.agg(["sum", "count"])
train_fe = train_fe.merge(grp, left_on="GroupID", right_index=True)
train_fe["FamSurvival"] = np.where(
    train_fe["count"] > 1,
    (train_fe["sum"] - train_fe.Survived) / (train_fe["count"] - 1), 0.5)
test_fe["FamSurvival"] = test_fe.GroupID.map(grp["sum"] / grp["count"]).fillna(0.5)

NUM = ["Age", "Fare", "FamilySize", "FamSurvival"]
CAT = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]
X, y = train_fe[NUM + CAT], train_fe.Survived

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])

# Winner of the 40-iteration random search (see version notes above)
gb = GradientBoostingClassifier(learning_rate=0.01, n_estimators=500, max_depth=3,
                                subsample=0.6, min_samples_leaf=3, random_state=42)
pipe = Pipeline([("pre", pre), ("m", gb)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
s = cross_val_score(pipe, X, y, cv=cv)
print(f"v6 CV accuracy: {s.mean():.4f} ± {s.std():.4f}")

pipe.fit(X, y)
preds = pipe.predict(test_fe[NUM + CAT])
pd.DataFrame({"PassengerId": test_fe.PassengerId, "Survived": preds}).to_csv(
    "submission_v6.csv", index=False)
print(f"submission_v6.csv written — predicted survival rate: {preds.mean():.3f}")

v6 CV accuracy: 0.8530 ± 0.0148


submission_v6.csv written — predicted survival rate: 0.364


## Version history

| Version | Change | CV | Leaderboard |
|---|---|---|---|
| v1 | Full pipeline | 0.844 | 0.76794 |
| v2 | Lean + rows deleted | 0.824 | 0.75837 |
| v3 | v1 minus Embarked | 0.835 | 0.76315 |
| v4 | v1 + fare per person | 0.847 | 0.75598 |
| v5 | + group survival, regularized GB | 0.850 | 0.78708 |
| v6 | v5 minus FarePP, tuned via search loop | 0.853 | — |

**Caveat:** after 40 configs tested against the same CV folds, part of the +0.3 over v5 may be selection luck — the leaderboard will arbitrate.
